In [26]:
from neo4j import GraphDatabase

# Demo database credentials
URI = "bolt://localhost:7687"

AUTH = ("neo4j", "conceptnorm")

# Connect to Neo4j database
driver = GraphDatabase.driver(URI, auth=AUTH)

In [27]:
from neo4j_graphrag.retrievers import HybridCypherRetriever
from neo4j_graphrag.embeddings import SentenceTransformerEmbeddings
from neo4j_graphrag.generation import GraphRAG
from neo4j_graphrag.llm import OllamaLLM

retrieval_query = """
MATCH (node:ObjectConcept)-[:HAS_DESCRIPTION]->(d:Description)

OPTIONAL MATCH (node)-[:HAS_ROLE_GROUP]->(rg:RoleGroup)
OPTIONAL MATCH (rg)-[:FINDING_SITE]->(site:ObjectConcept)

RETURN
    node.FSN AS FSN,
    node.sctid AS SNOMED_CT_ID,
    collect(DISTINCT d.term) AS descriptions,
    collect(DISTINCT {
        FSN: site.FSN,
        SNOMED_CT_ID: site.sctid
    }) AS finding_sites,
    score AS score
"""

embedder = SentenceTransformerEmbeddings(
    model="FremyCompany/BioLORD-2023"
)

retriever = HybridCypherRetriever(
    driver=driver,
    vector_index_name="snomed_concept_embeddings",
    fulltext_index_name="snomed_concept_fulltext",
    retrieval_query=retrieval_query,
    embedder=embedder,
)

llm = OllamaLLM(model_name="llama3.1", model_params={"temperature": 0.0})
rag = GraphRAG(
    retriever=retriever,
    llm=llm
)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4402.95it/s]


In [28]:
import ast


def print_retriever_results(result):
    for i, item in enumerate(result.items, 1):
        content = item.content

        # Remove <Record ...>
        content = content.removeprefix("<Record ").removesuffix(">")

        # Split the individual fields
        fsn_part, rest = content.split(" SNOMED_CT_ID=", 1)
        snomed_part, rest = rest.split(" descriptions=", 1)
        descriptions_part, rest = rest.split(" finding_sites=", 1)
        finding_sites_part, score_part = rest.rsplit(" score=", 1)

        # Safely convert Python-style values
        fsn = ast.literal_eval(fsn_part.removeprefix("FSN="))
        snomed_id = ast.literal_eval(snomed_part)
        descriptions = ast.literal_eval(descriptions_part)
        finding_sites = ast.literal_eval(finding_sites_part)
        score = float(score_part)

        # Remove empty finding sites
        finding_sites = [
            site for site in finding_sites
            if site.get("FSN") is not None
        ]

        print("\n" + "─" * 80)
        print(f"Result {i}  |  Score: {score:.4f}")
        print("─" * 80)

        print(f"{fsn}")
        print(f"SNOMED CT ID: {snomed_id}")

        print("\nDescriptions:")
        for description in descriptions:
            print(f"  • {description}")

        print("\nFinding sites:")
        if finding_sites:
            for site in finding_sites:
                print(
                    f"  • {site['FSN']} "
                    f"[{site['SNOMED_CT_ID']}]"
                )
        else:
            print("  • None")

In [29]:
query_text = "oncology|chest tumors|primary lung cancer"
retriever_result = retriever.search(query_text=query_text, top_k=3)

print_retriever_results(retriever_result)


────────────────────────────────────────────────────────────────────────────────
Result 1  |  Score: 1.0000
────────────────────────────────────────────────────────────────────────────────
Primary non-small cell lung cancer (disorder)
SNOMED CT ID: 1259727001

Descriptions:
  • Primary non-small cell lung cancer
  • Primary non-small cell lung cancer (disorder)

Finding sites:
  • Lung structure (body structure) [39607008]

────────────────────────────────────────────────────────────────────────────────
Result 2  |  Score: 1.0000
────────────────────────────────────────────────────────────────────────────────
Primary malignant neoplasm of main bronchus (disorder)
SNOMED CT ID: 93882009

Descriptions:
  • Primary malignant neoplasm of main bronchus (disorder)
  • Primary malignant neoplasm of main bronchus
  • Malignant neoplasm of main bronchus

Finding sites:
  • Lung structure (body structure) [39607008]
  • Main bronchus structure (body structure) [102297006]
  • Anatomical structu

In [30]:
query_text = "cardiovascular|arrhythmias|atrial fibrillation|with rapid ventricular response"
result = retriever.search(
    query_text=query_text,
    top_k=5,
)

print_retriever_results(result)


────────────────────────────────────────────────────────────────────────────────
Result 1  |  Score: 1.0000
────────────────────────────────────────────────────────────────────────────────
Atrial fibrillation with rapid ventricular response (disorder)
SNOMED CT ID: 120041000119109

Descriptions:
  • Atrial fibrillation with rapid ventricular response (disorder)
  • Atrial fibrillation with rapid ventricular response

Finding sites:
  • Cardiac conducting system structure (body structure) [24964005]
  • Atrial structure (body structure) [59652004]

────────────────────────────────────────────────────────────────────────────────
Result 2  |  Score: 0.9733
────────────────────────────────────────────────────────────────────────────────
Rapid atrial fibrillation (disorder)
SNOMED CT ID: 314208002

Descriptions:
  • Rapid atrial fibrillation (disorder)
  • Rapid atrial fibrillation

Finding sites:
  • Atrial structure (body structure) [59652004]
  • Cardiac conducting system structure (bod

In [31]:
def ask_question(question, top_k=3):
    """Ask a question and get an AI answer from the book database"""

    try:
        print(f"🔍 Question: {question}")

        # Search and get answer
        response = rag.search(
            query_text=question,
            retriever_config={"top_k": top_k}
        )

        print(f"🤖 Answer: {response.answer}")
        return response.answer

    except Exception as e:
        print(f"❌ Error: {e}")
        return None
    finally:
        driver.close()

In [32]:
ask_question("You are an expert clinical terminology normalizer for SNOMED CT. given the following input:oncology|chest tumors|primary lung cancer, please provide the most appropriate SNOMED CT concept and its details. Look at the context around the concept and provide a detailed explanation of why this concept is the most appropriate. Include any relevant SNOMED CT IDs, descriptions, and finding sites in your answer.", top_k=5)

🔍 Question: You are an expert clinical terminology normalizer for SNOMED CT. given the following input:oncology|chest tumors|primary lung cancer, please provide the most appropriate SNOMED CT concept and its details. Look at the context around the concept and provide a detailed explanation of why this concept is the most appropriate. Include any relevant SNOMED CT IDs, descriptions, and finding sites in your answer.
🤖 Answer: Based on the input "oncology|chest tumors|primary lung cancer", I would recommend the SNOMED CT concept "Primary malignant neoplasm of lung (disorder)" with the ID "93880001".

Here's a detailed explanation of why this concept is the most appropriate:

* The input mentions "oncology", which suggests a focus on cancer-related concepts.
* "Chest tumors" and "primary lung cancer" indicate a specific interest in lung cancer.
* The concept "Primary malignant neoplasm of lung (disorder)" is a broad term that encompasses various types of lung cancer, including primary no

'Based on the input "oncology|chest tumors|primary lung cancer", I would recommend the SNOMED CT concept "Primary malignant neoplasm of lung (disorder)" with the ID "93880001".\n\nHere\'s a detailed explanation of why this concept is the most appropriate:\n\n* The input mentions "oncology", which suggests a focus on cancer-related concepts.\n* "Chest tumors" and "primary lung cancer" indicate a specific interest in lung cancer.\n* The concept "Primary malignant neoplasm of lung (disorder)" is a broad term that encompasses various types of lung cancer, including primary non-small cell lung cancer and primary solid carcinoma of lung.\n* The concept has a high score of 1.0, indicating a strong match with the input.\n* The finding sites associated with this concept include "Intrathoracic organ (body structure)", "Respiratory organ (body structure)", "Lower respiratory tract structure (body structure)", and "Lung structure (body structure)", which are all relevant to lung cancer.\n* The des

In [19]:
from importlib import reload
import concept_normalisation.graphrag.graphrag_matcher
reload(concept_normalisation.graphrag.graphrag_matcher)
from concept_normalisation.graphrag.graphrag_matcher import GraphRAGMatcher
import json

def ask_question_with_matcher(question, top_k=3):
    matcher = GraphRAGMatcher()
    question = matcher.clean_query(question)
    result = matcher.search(question, top_k=top_k)
    try:
        print(json.dumps(result["matches"], indent=4))
    except Exception:
        print(result["error"])
    matcher.close()

In [17]:
# 129032061000119103
ask_question_with_matcher("gastrointestinal|intestinal disease|diarrhea|C. difficile colitis", top_k=5)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4439.69it/s]


[
    {
        "sctid": "5891000119102",
        "fsn": "Clostridium difficile diarrhea (disorder)",
        "reason": "Best match for 'diarrhea' and 'C. difficile colitis'",
        "score": 1.0
    },
    {
        "sctid": "16838111000119103",
        "fsn": "Recurrent colitis caused by Clostridium difficile (disorder)",
        "reason": "Matches 'colitis' and 'C. difficile'",
        "score": 0.9720192403518911
    },
    {
        "sctid": "16838151000119102",
        "fsn": "Recurrent diarrhea caused by Clostridium difficile (disorder)",
        "reason": "Matches 'diarrhea' and 'C. difficile'",
        "score": 0.9894613589420677
    },
    {
        "sctid": "1172958005",
        "fsn": "Intestinal infection caused by Clostridioides difficile (disorder)",
        "reason": "Matches 'C. difficile' but not specific to 'colitis' or 'diarrhea'",
        "score": 1.0
    },
    {
        "sctid": "1172959002",
        "fsn": "Extraintestinal infection caused by Clostridioides diff

In [20]:
# 120041000119109
ask_question_with_matcher("infectious diseases|systemic / other infections|sepsis", top_k=5)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4398.86it/s]


[
    {
        "sctid": "91302008",
        "fsn": "Sepsis (disorder)",
        "reason": "Direct match in the diagnosis string and high retrieval relevance",
        "score": 1.0
    },
    {
        "sctid": "10001005",
        "fsn": "Bacterial sepsis (disorder)",
        "reason": "High retrieval relevance and close match to the diagnosis string",
        "score": 0.9593823704980761
    },
    {
        "sctid": "238150007",
        "fsn": "Sepsis syndrome (disorder)",
        "reason": "High retrieval relevance and close match to the diagnosis string",
        "score": 0.9358848832122891
    },
    {
        "sctid": "418862001",
        "fsn": "Pediatric infectious diseases (qualifier value)",
        "reason": "Low retrieval relevance and not a direct match to the diagnosis string",
        "score": 1.0
    },
    {
        "sctid": "309934006",
        "fsn": "Infectious diseases department (environment)",
        "reason": "Low retrieval relevance and not a direct match to th

In [24]:
ask_question_with_matcher("oncology|chest tumors|primary lung cancer", top_k=5)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4195.91it/s]


Based on the given diagnosis string "oncology|chest tumors|primary lung cancer", I have ranked the candidate SNOMED concepts from most likely to least likely as follows:

1. sctid: 1259727001
FSN: Primary non-small cell lung cancer (disorder)
Reason: The FSN "Primary non-small cell lung cancer (disorder)" exactly matches the diagnosis string. It is also a very relevant concept, with a high score of 1.0.

2. sctid: 93882009
FSN: Primary malignant neoplasm of main bronchus (disorder)
Reason: The FSN "Primary malignant neoplasm of main bronchus (disorder)" is a close match to the diagnosis string, with the word "lung" implied by the word "main bronchus". It is also a very relevant concept, with a high score of 1.0.

3. sctid: 93880001
FSN: Primary malignant neoplasm of lung (disorder)
Reason: The FSN "Primary malignant neoplasm of lung (disorder)" is a match to the diagnosis string, but it is not as specific as the top two candidates. However, it is still a relevant concept, with a high s